# BMI Calculator & Healthy Lifestyle Recommendation Engine  
Smart Health Assistant – Module 3  
This notebook covers:
- BMI dataset exploration  
- Rule-based BMI calculation  
- Healthy Lifestyle dataset exploration  
- JSON-based recommendation engine  
- No ML models (because rule-based methods are medically accurate)


In [2]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


In [4]:
bmi = pd.read_csv(r"C:\Users\Rajesh Kinjarapu\Desktop\Smart Health Buddy\Datasets\BMI Dataset\500_Person_Gender_Height_Weight_Index.csv")
bmi.head()


,Gender,Height,Weight,Index
0,Male,174,96,4
1,Male,189,87,2
2,Female,185,110,4
3,Female,195,104,3
4,Male,149,61,3


In [6]:
bmi.info()
bmi.describe()
bmi['Gender'].value_counts()
bmi['Index'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Gender  500 non-null    object
 1   Height  500 non-null    int64 
 2   Weight  500 non-null    int64 
 3   Index   500 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 15.8+ KB


Index
5    198
4    130
2     69
3     68
1     22
0     13
Name: count, dtype: int64

In [8]:
def calculate_bmi(weight, height_cm):
    height_m = height_cm / 100
    bmi = weight / (height_m ** 2)
    return round(bmi, 2)

def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"


In [10]:
sample_bmi = calculate_bmi(70, 170)
sample_cat = bmi_category(sample_bmi)
sample_bmi, sample_cat


(24.22, 'Normal')

In [12]:
health = pd.read_csv(r"C:\Users\Rajesh Kinjarapu\Desktop\Smart Health Buddy\Datasets\Healthy Lifestyle\heart_2020_cleaned.csv")
health.head()


,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer
0,No,16.60,Yes,No,No,3.0,30.0,No,Female,55-59,White,Yes,Yes,Very good,5.0,Yes,No,Yes
1,No,20.34,No,No,Yes,0.0,0.0,No,Female,80 or older,White,No,Yes,Very good,7.0,No,No,No
2,No,26.58,Yes,No,No,20.0,30.0,No,Male,65-69,White,Yes,Yes,Fair,8.0,Yes,No,No
3,No,24.21,No,No,No,0.0,0.0,No,Female,75-79,White,No,No,Good,6.0,No,No,Yes
4,No,23.71,No,No,No,28.0,0.0,Yes,Female,40-44,White,No,Yes,Very good,8.0,No,No,No


In [14]:
health.info()
health.describe(include='all')
health['GenHealth'].value_counts()
health['Smoking'].value_counts()
health['AlcoholDrinking'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319795 entries, 0 to 319794
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   HeartDisease      319795 non-null  object 
 1   BMI               319795 non-null  float64
 2   Smoking           319795 non-null  object 
 3   AlcoholDrinking   319795 non-null  object 
 4   Stroke            319795 non-null  object 
 5   PhysicalHealth    319795 non-null  float64
 6   MentalHealth      319795 non-null  float64
 7   DiffWalking       319795 non-null  object 
 8   Sex               319795 non-null  object 
 9   AgeCategory       319795 non-null  object 
 10  Race              319795 non-null  object 
 11  Diabetic          319795 non-null  object 
 12  PhysicalActivity  319795 non-null  object 
 13  GenHealth         319795 non-null  object 
 14  SleepTime         319795 non-null  float64
 15  Asthma            319795 non-null  object 
 16  KidneyDisease     31

AlcoholDrinking
No     298018
Yes     21777
Name: count, dtype: int64

### Why no ML model for lifestyle?

Lifestyle improvement is NOT a prediction task.
It depends on:

- smoking habits  
- alcohol consumption  
- physical activity  
- sleep  
- general health  
- BMI  
- medical history  

These require **rule-based logic**, not classification.
Therefore, a recommendation engine is ideal.


In [19]:
lifestyle_tips = {
    "Smoking": {
        "Yes": ["Reduce smoking gradually", "Try nicotine replacement", "Consult support groups"]
    },
    "AlcoholDrinking": {
        "Yes": ["Limit alcohol to minimal intake", "Increase water intake"]
    },
    "PhysicalActivity": {
        "No": ["Start with 20 minutes walking daily", "Try simple yoga exercises"]
    },
    "SleepTime": {
        "Low": ["Get 7-8 hours sleep", "Avoid screens before bedtime"],
        "High": ["Avoid sleeping more than 9 hours", "Maintain a sleep cycle"]
    },
    "BMI": {
        "Underweight": ["Increase calorie intake", "Eat protein-rich foods"],
        "Normal": ["Maintain your current routine"],
        "Overweight": ["Avoid sugar", "Walk 30 mins daily"],
        "Obese": ["Follow calorie-deficit diet", "Consult dietitian"]
    },
    "GenHealth": {
        "Poor": ["Improve diet immediately", "Increase water intake", "Consult doctor"],
        "Fair": ["Increase sleep quality", "Improve lifestyle balance"],
        "Good": ["Keep your routine steady"],
        "Very good": ["Maintain positive habits"]
    },
    "Diabetic": {
        "Yes": ["Monitor glucose daily", "Avoid sweets", "Increase fiber intake"]
    },
    "HeartDisease": {
        "Yes": ["Avoid salt", "Do cardio exercises daily"]
    },
    "Stroke": {
        "Yes": ["Follow physiotherapy routines"]
    }
}

import json

json.dump(
    lifestyle_tips,
    open(r"C:\Users\Rajesh Kinjarapu\Desktop\Smart Health Assistant\models\lifestyle_tips.json", "w"),
    indent=4
)

print("Lifestyle tips JSON saved successfully!")



Lifestyle tips JSON saved successfully!


In [21]:
def get_lifestyle_recommendations(user):
    recs = []

    # BMI rules
    if "BMI" in user:
        bmi = user["BMI"]
        if bmi < 18.5:
            recs += lifestyle_tips["BMI"]["Underweight"]
        elif bmi < 25:
            recs += lifestyle_tips["BMI"]["Normal"]
        elif bmi < 30:
            recs += lifestyle_tips["BMI"]["Overweight"]
        else:
            recs += lifestyle_tips["BMI"]["Obese"]

    # Smoking
    if user.get("Smoking") == "Yes":
        recs += lifestyle_tips["Smoking"]["Yes"]

    # Alcohol
    if user.get("AlcoholDrinking") == "Yes":
        recs += lifestyle_tips["AlcoholDrinking"]["Yes"]

    # Physical activity
    if user.get("PhysicalActivity") == "No":
        recs += lifestyle_tips["PhysicalActivity"]["No"]

    # Sleep
    sleep = user.get("SleepTime")
    if sleep:
        if sleep < 6:
            recs += lifestyle_tips["SleepTime"]["Low"]
        elif sleep > 9:
            recs += lifestyle_tips["SleepTime"]["High"]

    # GenHealth
    gen = user.get("GenHealth")
    if gen in lifestyle_tips["GenHealth"]:
        recs += lifestyle_tips["GenHealth"][gen]

    # Diabetic
    if user.get("Diabetic") == "Yes":
        recs += lifestyle_tips["Diabetic"]["Yes"]

    # HeartDisease
    if user.get("HeartDisease") == "Yes":
        recs += lifestyle_tips["HeartDisease"]["Yes"]

    # Stroke
    if user.get("Stroke") == "Yes":
        recs += lifestyle_tips["Stroke"]["Yes"]

    return list(set(recs))


In [23]:
sample_user = {
    "BMI": 28,
    "Smoking": "Yes",
    "AlcoholDrinking": "No",
    "PhysicalActivity": "No",
    "SleepTime": 5,
    "GenHealth": "Fair",
    "Diabetic": "Yes",
    "HeartDisease": "No",
    "Stroke": "No"
}

get_lifestyle_recommendations(sample_user)


['Start with 20 minutes walking daily',
 'Consult support groups',
 'Reduce smoking gradually',
 'Increase fiber intake',
 'Walk 30 mins daily',
 'Avoid screens before bedtime',
 'Get 7-8 hours sleep',
 'Avoid sugar',
 'Increase sleep quality',
 'Try nicotine replacement',
 'Monitor glucose daily',
 'Improve lifestyle balance',
 'Try simple yoga exercises',
 'Avoid sweets']

In [3]:
import os
os.makedirs("../backend", exist_ok=True)


In [5]:
backend_code = """
# FULL FLASK BACKEND CODE HERE
"""

with open("../backend/app.py", "w") as f:
    f.write(backend_code)


In [7]:
!pip install flask flask-cors numpy scikit-learn xgboost


In [15]:
import requests

data = {
    "Pregnancies": 2,
    "Glucose": 120,
    "BloodPressure": 70,
    "SkinThickness": 20,
    "Insulin": 79,
    "BMI": 28.5,
    "DiabetesPedigreeFunction": 0.47,
    "Age": 45
}

requests.post("http://127.0.0.1:5000/predict/diabetes", json=data).json()


{'diabetes_risk': 0}

In [17]:
import requests

heart_data = {
    "age": 63,
    "sex": 1,
    "cp": 3,
    "trestbps": 145,
    "chol": 233,
    "fbs": 1,
    "restecg": 0,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 2.3,
    "slope": 0,
    "ca": 0,
    "thal": 1
}

requests.post("http://127.0.0.1:5000/predict/heart", json=heart_data).json()


{'heart_risk': 1}

In [19]:
requests.post(
    "http://127.0.0.1:5000/bmi",
    json={"weight": 70, "height": 170}
).json()


{'bmi': 24.22, 'category': 'Normal'}

In [21]:
user = {
    "BMI": 28,
    "Smoking": "Yes",
    "AlcoholDrinking": "No",
    "PhysicalActivity": "No",
    "SleepTime": 5,
    "GenHealth": "Fair",
    "Diabetic": "Yes",
    "HeartDisease": "No",
    "Stroke": "No"
}

requests.post(
    "http://127.0.0.1:5000/recommend/lifestyle",
    json=user
).json()


{'recommendations': ['Avoid sugar',
  'Walk 30 mins daily',
  'Reduce smoking gradually',
  'Try nicotine replacement',
  'Consult support groups',
  'Start with 20 minutes walking daily',
  'Try simple yoga exercises',
  'Get 7-8 hours sleep',
  'Avoid screens before bedtime',
  'Increase sleep quality',
  'Improve lifestyle balance',
  'Monitor glucose daily',
  'Avoid sweets',
  'Increase fiber intake']}